In [ ]:
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from matplotlib.lines import Line2D

from climate_attitudes.cli.visualisation.directional_differential import (
    plot_ranked_differentials,
)
from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import DIVERGING_CMAP, configure_mpl
from ising import Ising

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

In [ ]:
# fig, ax = plt.subplots(figsize=(5, 6), constrained_layout=True)

# # Scatter means

# mean_diffs_flat = mean_diff[np.triu_indices_from(mean_diff, k=1)]
# ci_flat = ci[np.triu_indices_from(ci, k=1)]
# ax.scatter(
#     mean_diffs_flat,
#     np.arange(n * (n - 1) // 2),
#     color="k",
#     s=14,
#     zorder=5,
#     label="Mean effect difference",
# )

# # Show ci interval as red shaded region
# marker, _, bar = ax.errorbar(
#     mean_diffs_flat,
#     np.arange(n * (n - 1) // 2),
#     xerr=np.array([ci_flat, ci_flat]),
#     ls="none",
#     zorder=3,
#     color="tab:red",
#     label="95% CI",
# )
# plt.setp(bar[0], capstyle="round")
# marker.set_fillstyle("none")
# bar[0].set_alpha(0.5)
# bar[0].set_linewidth(5)

# # Draw 0.0 as dashed
# ax.axvline(x=0, linestyle="dashed", linewidth=0.75, color="gray", zorder=1)

# ylabels = []
# colnames = [ds_spec.RENAME.get(colname, colname) for colname in labels]
# for i in range(n - 1):
#     for j in range(i + 1, n):
#         c1 = colnames[i]
#         c2 = colnames[j]
#         ylabels.append(f"{c1} -> {c2}")

# ax.set_yticks(np.arange(len(ylabels)), ylabels)

# ax.legend(
#     ncol=2,
#     loc="lower center",
#     bbox_to_anchor=(0.5, 1.0),
#     # fontsize=8,
#     # handlelength=1,
#     # columnspacing=0.5,
#     # labelspacing=0.2,
#     frameon=False,
# )

# plt.show();


Plot directional differentials as circles on grid. Colour blue if positive, red if negative. Set circle size based on size of CI.

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(
    config,
    name="reduced_no_imputation",
    with_imputation=False,
    verbose=False,
)

labels = dataset.schema.get_short_names(kind="measurement")

measurements = np.load(
    Path(
        "../reports/thesis/results/data/model/bootstrapped_fit/ising_no_use_covariates_no_structure.npz"
    )
)["params"]
Ys = np.load(
    Path(
        "../reports/thesis/results/data/model/bootstrapped_fit/ising_no_use_covariates_no_structure.npz"
    )
)["Y"]
js = measurements[:, 8:].reshape((-1, 8, 8))
diffs = js - np.swapaxes(js, 1, 2)

mean_diff = diffs.mean(axis=0)
ci = 1.96 * np.std(diffs, axis=0, ddof=1) / np.sqrt(diffs.shape[0])

# mean_diff = mean_diff - mean_diff.T
# ci = ci + ci.T

# n = measurements.shape[-1]

In [ ]:
dataset.response.collect().sort(by=("participant_id", "wave")).head()

In [ ]:
dataset.indices.collect().sort(by=("participant_id", "wave")).head()

In [ ]:
measurements.shape

In [ ]:
Ys.shape[2]

Identify which interactions are insignificant

In [ ]:
# Estimate variance around true parameter using inverse fisher information
n = Ys.shape[-1]
Is = np.empty((measurements.shape[0], n**2, n**2), dtype=np.float64)
for i, params in enumerate(measurements):
    h = params[:n]
    j = params[n:].reshape((n, n))
    adj = np.full((n, n), fill_value=True, dtype=bool)
    X = np.ones((Ys.shape[1], Ys.shape[2]), dtype=np.float64)
    Is[i] = (
        Ising.time_series_nll_hessian_sync(Ys[i], X, h, j, adj)[n:][:, n:]
        * np.eye(n**2)
        * Ys.shape[1]
        * Ys.shape[2]
    )

mean_I = Is.mean(axis=0)

I_inv = np.linalg.inv(mean_I)
j_ci = 1.96 * np.sqrt(I_inv.diagonal().reshape((n, n))) / np.sqrt(measurements.shape[0])

j_mean = js.mean(axis=0)
significant = (j_mean + j_ci < 0) | (j_mean - j_ci > 0)
significant

In [ ]:
1.96 * js.std(axis=0, ddof=1) / np.sqrt(300)

Which interaction pairs are unidirectional? (significant interaction in only one direction)

In [ ]:
unidirectional = np.logical_xor(significant, significant.T)
unidirectional

Which are significant in neither direction?

In [ ]:
null = ~significant & ~significant.T
null

Which interaction pairs are symmetric? (equal, non-zero interactions in either direction)

In [ ]:
equal_effect = ~((mean_diff - ci > 0) | (mean_diff + ci < 0))
symmetric = (significant & significant.T) & equal_effect
symmetric

Which are asymmetric? (non-equal, both significant)

In [ ]:
asymmetric = (significant & significant.T) & ~equal_effect
asymmetric

In [ ]:
labels

In [ ]:
plot_ranked_differentials(mean_diff, ci, labels);

In [ ]:
mean_diff

In [ ]:
measurements[1]

In [ ]:
ci

In [ ]:
cmap = DIVERGING_CMAP

In [ ]:
mean_diff[::-1]

In [ ]:
# mean_diff[6, :], mean_diff[:, 6] = mean_diff[:, 6], mean_diff[6, :]

In [ ]:
sort_idxes = np.argsort(mean_diff.mean(axis=1))
mean_diff = mean_diff[sort_idxes][:, sort_idxes]
ci = ci[sort_idxes][:, sort_idxes]
labels = [labels[i] for i in sort_idxes]

In [ ]:
labels

In [ ]:
ci.min(), ci.max()

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5), constrained_layout=True)
ax.set_aspect("equal")

# Show mean differential using colour (red -ve, blue +ve); CIs with proportional size
X, Y = np.meshgrid(np.arange(mean_diff.shape[0]), np.arange(mean_diff.shape[0]))
colours = cmap(mean_diff * 0.5 / (0.2) + 0.5)

# min size non-zero CI sets max marker size. Larger CIs are smaller. double CI should
#   be half size (radius?).
min_ci = np.sort(ci.flatten())[ci.shape[0]]
sizes = ci.copy()
sizes[np.diag_indices_from(sizes)] = np.inf
sizes = min_ci / sizes

# Re-scale so min-size CI is 300 points
sizes *= 500

# sizes = (ci / 0.025) * 300
ax.scatter(
    X.flatten(), Y.flatten(), s=sizes.flatten(), c=colours.ravel().reshape((-1, 4))
)

# Set tick labels
ax.set_xticks(
    np.arange(8), labels, rotation=30, horizontalalignment="right", fontsize=9
)
ax.set_yticks(np.arange(8), labels, rotation=0, horizontalalignment="right", fontsize=9)

# Axis labels
ax.set_ylabel(r"From", rotation=90, labelpad=20, fontsize=12)
ax.set_xlabel(r"To", fontsize=12)

# Hide tick markers, axis frame/spines
ax.tick_params(axis="both", which="both", length=0)
for spine in ax.spines.values():
    spine.set_visible(False)

# == Legends
# Colourbar for mean differential
cbar = fig.colorbar(
    cm.ScalarMappable(norm=mcolors.Normalize(vmin=-0.2, vmax=0.2), cmap=DIVERGING_CMAP),
    shrink=0.65,
    aspect=25,
    ax=ax,
    pad=0.1,
)
cbar.set_ticks(np.linspace(-0.2, 0.2, 5))
cbar.ax.set_title(r"$\delta_J$", pad=18)
# cbar.ax.set_ylabel(r"Mean directional differential", labelpad=10)


# Size chart for CI
# 0.01, 0.02, 0.03
ci_legend_vals = np.array([0.0025, 0.005, 0.01])
ci_legend_labels = [r"$2.5\times 10^{-3}$", r"$5\times 10^{-3}$", r"$10^{-3}$"]
ci_legend_sizes = (min_ci / ci_legend_vals) * 500
ci_legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markersize=s**0.5,  # scatter size s is in points²
        label=f"{v:.4f}",
        color="black",
        markerfacecolor="white",
    )
    for v, s in zip(ci_legend_vals, ci_legend_sizes, strict=True)
]
ax.legend(
    ncol=3,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    handles=ci_legend_handles,
    title="95% CI",
    frameon=False,
    labelspacing=1.0,
)

fig.savefig(
    "../reports/thesis/results/figures/model/directional_differentials/heatmap.pdf",
    bbox_inches="tight",
)
fig.savefig(
    "../reports/thesis/results/figures/model/directional_differentials/heatmap.png",
    bbox_inches="tight",
)

In [ ]:
ci

In [ ]:
mean_diff